# Fanken Adapter

## 2.a training on ABSA dataset (IND)

In [ ]:
from franken_adapter.franken import instruction_tuning

save_path = "outputs/models/franken/2a/final_model.pt"
json_path = "hotel_dataset/hotel_aste_train_augmented_noreasoning.json"
model_name = "Qwen/Qwen2.5-0.5B"
device = "cuda"

instruct_model = instruction_tuning(model_name, json_path, 
                                    num_epochs=5, batch_size=32, lr=5e-4, 
                                    save_path=None, 
                                    circuit_path=None)

## 2.b Training embedding on Sundanese Wikipedia

### Preprocessing

In [ ]:
from franken_adapter.utils import filter_and_save_non_english_articles

filter_and_save_non_english_articles(
    input_dir="wiki_dataset/suwiki_extracted/AA",
    output_file="wiki_dataset/suwiki_filtered.jsonl",
    min_word_count=10
)

In [ ]:
from franken_adapter.franken import embedding_training

embedding_model = embedding_training(model_name="Qwen/Qwen2.5-0.5B",
                                    num_epochs=5, batch_size=32, lr=5e-4,
                                    jsonl_path="wiki_dataset/suwiki_filtered.jsonl",
                                    save_path=None
                                    )

## 3.a Combine

 combine new embeddings with instruction-tuned transformer body as the
Franken-Adapter

In [ ]:
from franken_adapter.franken import merge_instruction_with_embedding

franken_model = merge_instruction_with_embedding(
    base_model_name="Qwen/Qwen2.5-0.5B",
    instruct_weights_path="outputs/models/franken/2a/full_model_10epochs.pt",
    embedding_weights_path="outputs/models/franken/2b/full_model_epochs-10_samples-full.pt",
    output_path="outputs/models/franken/3/franken_adapter_full_model_embed-samples-full.pt",
    device="cuda"
)

### Test model without finetuning

In [1]:
import torch
from transformer_lens import HookedTransformer

base_model_name="Qwen/Qwen2.5-0.5B"
franken_weights_path="outputs/models/franken/3/franken_adapter_full_model_embed-samples-full.pt"
device="cuda"

franken_model = HookedTransformer.from_pretrained(base_model_name, device=device)
state_dict_franken = torch.load(franken_weights_path, map_location=device)
franken_model.load_state_dict(state_dict_franken)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer


<All keys matched successfully>

In [2]:
franken_model.generate(["tempatna alus . kolam renangna beresih . [A] [O] [S]"], 
               max_new_tokens=50,
               stop_at_eos=True,
               return_type="str")

  0%|          | 0/50 [00:00<?, ?it/s]

'tempatna alus . kolam renangna beresih . [A] [O] [S] [A] tempin [ [ [ [ [ [ [ [ [ctic [ [SSEP] tvatas [ih [ening enough [ [ [ [ [ [ [ ['

## 3.b Finetune on ABSA dataset (SUNDANESE)

In [ ]:
from franken_adapter.franken import finetune_franken_adapter

final_model = finetune_franken_adapter(
    base_model_name="Qwen/Qwen2.5-0.5B",
    franken_weights_path="outputs/models/franken/3/franken_adapter_full_model_embed-samples-full.pt",
    json_data_path="hotel_dataset/sunda/hotel_aste_train_augmented_noreasoning_sample.json",
    num_epochs=5, batch_size=32, lr=5e-4,
    freeze_embedding=True,
    save_path="outputs/models/franken/3b/full_model_embed-samples-full_finetune-samples-full_finetune-epochs-5_freeze_embed.pt"
)

### Test model with finetuning

In [3]:
import torch
from transformer_lens import HookedTransformer

base_model_name="Qwen/Qwen2.5-0.5B"
final_weights_path="outputs/models/franken/3b/full_model_embed-samples-full_finetune-samples-full_finetune-epochs-5_freeze_embed.pt"
device="cuda"

final_model = HookedTransformer.from_pretrained(base_model_name, device=device)
state_dict_final = torch.load(final_weights_path, map_location=device)
final_model.load_state_dict(state_dict_final)

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer


<All keys matched successfully>

In [4]:
final_model.generate(["tempatna alus . kolam renangna beresih . [A] [O] [S]"], 
               max_new_tokens=100,
               stop_at_eos=True,
               return_type="str")

  0%|          | 0/100 [00:00<?, ?it/s]

'tempatna alus . kolam renangna beresih . [A] [O] [S] [A] tempatna [O] alus [S] positive [SSEP] [A] kolam renangna [O] beresih [S] positive'